# 01 · Data exploration

**Goal:** understand the *structure* of a series before choosing a model. Every model family later in the lab exploits one or more of the components we find here.

A classical mental model is the additive decomposition

$$y_t = T_t + S_t + R_t,$$

where $T_t$ is the trend, $S_t$ the seasonal component(s) and $R_t$ the remainder. When the seasonal amplitude grows with the level, the decomposition is multiplicative, $y_t = T_t \cdot S_t \cdot R_t$, which becomes additive after a $\log$.

**Datasets**
- **Jena climate** (Max Planck Institute for Biogeochemistry): 14 weather variables, hourly, 2009–2016. Multivariate, two seasonalities (day and year). Target: air temperature `T (degC)`.
- **Airline passengers** (Box & Jenkins): monthly, 1949–1960. Univariate, trend plus multiplicative yearly seasonality. This is the textbook SARIMA example.

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 4)

In [ ]:
from tsforecasting.data.data_loader import load_airline, load_jena, temporal_split
from tsforecasting.features.build_features import TARGET

try:
    df = load_jena()
except FileNotFoundError:  # first run: download + clean + features
    from tsforecasting.data import make_dataset
    from tsforecasting.features import build_features
    make_dataset.main(); build_features.main()
    df = load_jena()
y = df[TARGET]
airline = load_airline()
df.shape, airline.shape

## 1. A first look

Always plot the raw data at several time scales: the full span reveals trend and yearly cycles, a month reveals daily cycles.

In [ ]:
cols = ["T (degC)", "p (mbar)", "rho (g/m**3)"]
fig, axes = plt.subplots(2, 1, figsize=(12, 7))
df[cols].plot(subplots=False, ax=axes[0], lw=0.5, title="Full span (2009-2016)")
df[cols].iloc[: 24 * 30].plot(ax=axes[1], title="First 30 days")
fig.tight_layout()

In [ ]:
df.describe().T.round(2)

## 2. Cleaning and feature engineering (done in `src/`)

`tsforecasting.data.make_dataset` keeps on-the-hour records and clips the `-9999` wind-speed sentinels to 0. It also linearly interpolates the 77 hours in which the station recorded nothing.

> **Pitfall worth remembering.** The TensorFlow tutorial sub-samples with `df[5::6]`. The raw file has missing 10-minute rows, so that slice silently drifts off the hour in late 2016, and an hourly re-index then produces ~1,600 constant hours, right in the test period. We select `index.minute == 0` instead.

### Wind: polar to Cartesian
A direction in degrees is a bad input because $0^\circ$ and $360^\circ$ are the same direction yet numerically far apart. With speed $v$ and angle $\theta$:

$$W_x = v\cos\theta, \qquad W_y = v\sin\theta.$$

In [ ]:
from tsforecasting.data.make_dataset import INTERIM_FILE
from tsforecasting.utils.paths import data_interim_dir

raw = pd.read_parquet(data_interim_dir(INTERIM_FILE))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist2d(raw["wd (deg)"], raw["wv (m/s)"], bins=(50, 50), vmax=400)
axes[0].set(xlabel="direction [deg]", ylabel="speed [m/s]", title="Polar: hard to learn")
axes[1].hist2d(df["Wx"], df["Wy"], bins=(50, 50), vmax=400)
axes[1].set(xlabel="$W_x$ [m/s]", ylabel="$W_y$ [m/s]", title="Cartesian: smooth")
axes[1].set_aspect("equal")
fig.tight_layout()

### Which periodicities matter? The periodogram

The discrete Fourier transform $\hat y(f) = \sum_t y_t e^{-2\pi i f t}$ decomposes the series into frequencies. Peaks in $|\hat y(f)|$ reveal cycles. With frequencies expressed in cycles per year we expect peaks at $f=1$ (yearly) and $f\approx 365$ (daily).

In [ ]:
from tsforecasting.features.build_features import HOURS_PER_YEAR
from tsforecasting.visualization.visualize import plot_periodogram

ax = plot_periodogram(y, samples_per_unit=HOURS_PER_YEAR, unit="year")
ax.set_xticks([1, 365.2524], labels=["1/year", "1/day"])
ax.set_ylim(0, 4e5);

### Encoding time with sine/cosine

A timestamp in hours $t$ becomes, for each period $P\in\{24,\ 8765.8\}$,

$$\sin\!\left(\tfrac{2\pi t}{P}\right),\qquad \cos\!\left(\tfrac{2\pi t}{P}\right).$$

The pair is needed: $\sin$ alone maps 6 am and 6 pm to the same value. These *calendar features* are known in advance, which makes them **future-known covariates**. We'll feed them to skforecast and TimesFM later.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
df[["Day sin", "Day cos"]].iloc[:48].plot(ax=axes[0], title="time of day (48 h)")
df[["Year sin", "Year cos"]].iloc[: 24 * 365 * 2 : 24].plot(ax=axes[1], title="time of year (2 years)")
fig.tight_layout()

## 3. Decomposition with STL

STL (Seasonal-Trend decomposition using LOESS) estimates $T_t$ and $S_t$ with local regressions and is robust to outliers. On four weeks of hourly temperature with period $m=24$:

In [ ]:
from tsforecasting.visualization.visualize import plot_decomposition

plot_decomposition(y.iloc[24 * 180 : 24 * 208], period=24);

The airline series shows **multiplicative** seasonality: the swings grow with the level. A $\log$ makes them roughly constant, which is the classical *variance-stabilizing* transform. Box-Cox generalizes it: $y^{(\lambda)} = (y^\lambda-1)/\lambda$, with $\lambda\to0$ giving $\log y$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
airline.plot(ax=axes[0], title="airline passengers")
np.log(airline).plot(ax=axes[1], title="log(airline passengers)")
fig.tight_layout()
plot_decomposition(np.log(airline), period=12);

## 4. Autocorrelation: how does the past inform the future?

The autocorrelation function at lag $k$,

$$\rho_k = \frac{\operatorname{Cov}(y_t, y_{t-k})}{\operatorname{Var}(y_t)},$$

measures linear memory. The **partial** autocorrelation $\phi_{kk}$ is the correlation between $y_t$ and $y_{t-k}$ after removing the effect of lags $1,\dots,k-1$.

Rules of thumb for ARIMA identification:
- AR($p$): the PACF cuts off after lag $p$ and the ACF decays.
- MA($q$): the ACF cuts off after lag $q$ and the PACF decays.

Seasonality appears as spikes at multiples of $m$.

In [ ]:
from tsforecasting.visualization.visualize import plot_acf_pacf

plot_acf_pacf(y.iloc[-24 * 60 :], lags=72).suptitle("Jena temperature (last 60 days)", y=1.02);
plot_acf_pacf(np.log(airline).diff().diff(12).dropna(), lags=36).suptitle(
    r"airline: $\nabla\nabla_{12}\log y_t$", y=1.02);

## 5. Stationarity

ARIMA-type models assume (weak) stationarity: constant mean and an autocovariance that depends only on the lag. Two complementary tests:

| test | $H_0$ | reject $H_0$ (p < 0.05) means |
|---|---|---|
| ADF (Augmented Dickey-Fuller) | unit root (non-stationary) | stationary |
| KPSS | stationary | non-stationary |

Differencing $\nabla y_t = y_t - y_{t-1}$ removes a stochastic trend and seasonal differencing $\nabla_m y_t = y_t - y_{t-m}$ removes a seasonal one. These are the $d$ and $D$ in SARIMA$(p,d,q)(P,D,Q)_m$.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

def stationarity(series):
    s = series.dropna()
    return {"ADF p": adfuller(s)[1], "KPSS p": kpss(s, nlags="auto")[1]}

la = np.log(airline)
pd.DataFrame({
    "log y": stationarity(la),
    "∇ log y": stationarity(la.diff()),
    "∇∇₁₂ log y": stationarity(la.diff().diff(12)),
}).T.round(3)

## Takeaways
- Jena temperature has **two seasonalities** (24 h and ~8766 h) plus weather-driven noise. Multivariate covariates (pressure, humidity) carry information.
- Airline needs a $\log$, one regular and one seasonal difference. That is the "airline model" SARIMA$(0,1,1)(0,1,1)_{12}$, which we fit in notebook 03.
- Next: before fitting anything clever, define **how we evaluate** and **what we must beat** (notebook 02).